<div style="background-color:#4C4C9D; padding: 18px; border-radius: 3px; text-align:center; font-size:2.5em; font-weight:bold; color:#222; margin-bottom:25px; letter-spacing:1px;">
Predicting Donor Response for Social Good 
</div>

# <h2 style="border-bottom: 4px solid #4C4C9D; width:fit-content; margin: 0 auto 25px auto; padding-bottom:6px; font-weight:bold;">Model Selection</h2>

_Data Mining II 2025/2026_

Project by:
Francisco Gomes (20221810), Margarida Marchão (20221901), Marta Alves (20221890), Pedro Coimbras (20211573)


## <h3 style="border-bottom: 4px solid #4C4C9D; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">1. Imports and data loading</h3>

This notebook makes use of the previously transformed and processed datasets to train and evaluate different prediction models.


In [ ]:
"""Importing the libraries needed for data handling, preprocessing, modeling, and evaluation"""
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import re
import ast

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    StackingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import cross_val_score
from IPython.display import display
import optuna
from sklearn.exceptions import ConvergenceWarning

"""Adding the project root to the Python path so notebook imports work correctly"""
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils_modeling import build_model

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)



#warnings.filterwarnings("ignore", category=ConvergenceWarning, module="sklearn")  # Run to remove warnings
#warnings.resetwarnings()  # Run to add warnings


In [19]:
"""Loading the features datasets"""
train_data = pd.read_csv(PROJECT_ROOT / "project_data" / "X_train_clean.csv", index_col=0)
val_data = pd.read_csv(PROJECT_ROOT / "project_data" / "X_val_clean.csv", index_col=0)
test_data = pd.read_csv(PROJECT_ROOT / "project_data" / "X_test_clean.csv", index_col=0)

'''Loading the y datasets'''
y_train = pd.read_csv(PROJECT_ROOT / "project_data" / "y_train_clean.csv", index_col=0).squeeze()
y_val   = pd.read_csv(PROJECT_ROOT / "project_data" / "y_val_clean.csv",   index_col=0).squeeze()



## <h3 style="border-bottom: 4px solid #4C4C9D; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2. Training the final model</h3>

Training and tuning the previously defined models

### Importing the Chosen Model

The following cell automatically retrieves the last entry in our Results Keeping markdown, which contains the name and parameters of the model

In [ ]:
# Open results history
with open("results_history.md", "r") as f:
    content = f.read()

# Get last entry (after last ---)
last_entry = content.split("---")[-1]

# Parse fields
model_name  = re.search(r"\*\*Model:\*\* (.+)", last_entry).group(1).strip()
f1_score_   = re.search(r"\*\*F1 Score \(val\):\*\* (.+)", last_entry).group(1).strip()
params_str  = re.search(r"\*\*Parameters:\*\* (.+)", last_entry).group(1).strip()
params_dict = ast.literal_eval(params_str)

print(f"Loaded from results_history.md:")
print(f"  Model:  {model_name}")
print(f"  F1:     {f1_score_}")
print(f"  Params: {params_dict}")

# Build and train
import ast
Model = build_model(model_name, params_dict)

### Training the Chosen Model on the whole dataset 
> **Note** : Although the dataset was split into train and validation to produce the models, the whole dataset may be used in the final submission, as it acts as a richer dataset for Kaggle.

In [ ]:
# Concatonating train and validation data
full_X = np.concatenate([train_data, val_data])  
full_y = np.concatenate([y_train, y_val])


# Cross-validation on training set
cv_scores = cross_val_score(Model, full_X, full_y, cv=5, scoring='f1')
print(f"CV F1 scores: {cv_scores}")
print(f"CV F1 mean: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# Final fit on full training set
Model.fit(train_data, y_train)

# Final evaluation on held-out validation set (used ONLY here)
val_f1 = f1_score(y_val, Model.predict(val_data))
print(f"Final validation F1: {val_f1:.4f}")

# Align test_data columns with train_data columns
test_data_aligned = test_data[train_data.columns]
predModel = Model.predict(test_data_aligned)
Model_predictions = pd.DataFrame(predModel, index=test_data_aligned.index, columns=["TARGET_B"])
Model_predictions


,TARGET_B
CONTROL_NUMBER,
122653,0
184239,1
5172,0
135377,0
62119,0
...,...
54438,0
122194,0
106603,0


## <h3 style="border-bottom: 4px solid #4C4C9D; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">3. Exporting final predictions</h3>


In [22]:
Model_predictions.to_csv((PROJECT_ROOT / "project_data" / "Predictions.csv"))